# 00 · Setup

Questo notebook controlla che l'ambiente sia pronto per le esercitazioni. Eseguilo **prima della lezione**: bastano un paio di minuti.

**Su Google Colab** serve solo un account Google. La cella di setup scarica il repository e installa le librerie che mancano.

**In locale** devi aver già clonato il repository e installato le librerie (vedi il `README.md`). La cella di setup trova il repository da sola e non installa niente.

Esegui le celle in ordine, dall'alto in basso, con `Shift + Invio`.

## 1. Setup

La stessa cella apre tutti i notebook delle esercitazioni. Su Colab, la prima volta richiede circa un minuto.

In [ ]:
# Cella di setup: eseguila per prima, ogni volta che apri il notebook.
import subprocess, sys
from pathlib import Path

REPO = "salvatore-drago/nids-ml-lab"

if "google.colab" in sys.modules:
    root = Path("/content/nids-ml-lab")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        try:  # repository privato: token nei Secrets di Colab, con nome GITHUB_TOKEN
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(root)]).returncode:
            raise RuntimeError("Clone del repository non riuscito: controlla la connessione o l'accesso.")
    else:
        subprocess.run(["git", "-C", str(root), "pull", "-q"])
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nidslab").is_dir()), None)
    if root is None:
        raise RuntimeError("Apri il notebook dalla cartella notebooks/ del repository nids-ml-lab.")

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import nidslab as nl

nl.ensure_packages()
nl.set_style()
print("Repository:", root)

## 2. Controllo dell'ambiente

Ogni riga deve avere un ✓. Un ! è un avviso, un ✗ è un problema da risolvere prima della lezione.

In [ ]:
nl.check_environment();

## 3. Uno sguardo ai dati

Le esercitazioni usano **BCCC-CIC-IDS2017**: flussi di rete catturati in cinque giorni di luglio 2017, dal lunedì al venerdì, in una rete in cui sono stati lanciati attacchi diversi in giorni diversi. Ogni riga è un flusso (una conversazione tra due host), descritto da circa 115 caratteristiche statistiche.

Qui carichiamo il campione della prima esercitazione. Provenienza e preparazione dei dati sono descritte in `data/README.md`.

In [ ]:
df = nl.load_static()
print(f"{len(df):,} flussi, {df.shape[1]} colonne, {df.memory_usage(deep=True).sum() / 1e6:.0f} MB in memoria")
print(f"Dal {df['timestamp'].min():%d/%m/%Y %H:%M} al {df['timestamp'].max():%d/%m/%Y %H:%M}")
df[["timestamp", "src_ip", "dst_ip", "dst_port", "protocol", "duration",
    "packets_count", "total_payload_bytes", "label"]].head()

Quanti flussi ci sono per ogni classe? La scala orizzontale è logaritmica: ogni tacca vale dieci volte la precedente.

In [ ]:
import matplotlib.pyplot as plt

counts = df["label"].value_counts().sort_values()
colors = [nl.BENIGN_COLOR if lab == "Benign" else nl.ATTACK_COLOR for lab in counts.index]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(counts.index, counts.values, color=colors, height=0.7)
ax.set_xscale("log")
ax.grid(axis="y", visible=False)
for y, v in enumerate(counts.values):
    ax.text(v * 1.15, y, nl.fmt_int(v), va="center", fontsize=8, color=nl.INK_SECONDARY)
ax.set_xlim(right=counts.max() * 6)
ax.set_xlabel("Numero di flussi (scala logaritmica)")
ax.set_title("Flussi per classe nel campione della Lezione 1", loc="left")
plt.show()

print(f"Attacchi: {df['label_binary'].mean():.1%} dei flussi")

Il traffico benigno è la grande maggioranza, e alcune classi d'attacco hanno appena qualche decina di esempi. Ci torneremo nelle esercitazioni.

## Se qualcosa non va

**Su Colab**
- *La cella di setup dà errore:* menu **Runtime → Riavvia sessione**, poi riesegui dall'inizio.
- *La sessione si è disconnessa:* succede dopo un periodo di inattività. Riesegui le celle dall'inizio: i dati si ricaricano in pochi secondi.
- *Vuoi tenere le tue modifiche:* **File → Salva una copia in Drive**. Senza questo passo, quando chiudi la sessione le modifiche si perdono.

**In locale**
- *Mancano delle librerie:* dalla radice del repository, con il venv attivo, `pip install -r requirements.txt`.
- *Mancano i dati:* aggiorna il repository con `git pull`.
- *Il notebook non trova il repository:* aprilo dalla cartella `notebooks/` e scegli come kernel il venv del progetto.